In [ ]:
import random
from datasets import load_dataset, get_dataset_config_names
from tqdm import tqdm

# ==========================================================================
# 📚 데이터셋 소개 및 목표
# --------------------------------------------------------------------------
# [데이터셋 제목]: 중영어-대만어 군사 용어 번역 데이터셋
# [데이터셋 의미]: 이 데이터셋은 군사 분야의 영문 용어와 이를 대만에서 사용되는 번체 중국어(zh-TW)로 번역한 예시들을 담고 있습니다.
# [실습 목표]: 우리가 직접 데이터에서 특정 패턴을 찾고, '만약 내가 이 용어 검색 엔진을 만든다면?' 이라는 질문에 답하는 과정을 통해 데이터 분석의 재미를 느끼는 것이 목표입니다. 🚀
# ==========================================================================


# 📌 설정 변수
DATASET_NAME = "LichtLiu/en-zh_TW-military-terms"
SPLIT_TO_LOAD = "test" # 주로 테스트 셋으로 로딩하여 예제 크기를 줄입니다.
SAMPLE_COUNT = 10      # 테스트에 사용할 샘플 개수 (너무 크면 느려요!)
print(f"💖 AI 튜터가 준비한 데이터셋: {DATASET_NAME}")
print(f"✨ 로드할 샘플 개수: {SAMPLE_COUNT}개")
print("--------------------------------------------------")


# 1. 데이터셋 설정 확인 및 로딩
# --------------------------------------------------
print("\n[Step 1] 데이터 로드 준비 및 스트리밍 테스트...")

# 먼저 사용 가능한 Config 목록을 확인하는 습관을 들여봅시다! (필수 코딩 스킬이에요!)
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 성공! 사용 가능한 Config 목록: {configs}")
    # 이 데이터셋은 config가 단순하므로 첫 번째 config를 사용합니다.
    selected_config = configs[0]
except Exception as e:
    print(f"ℹ️ Config 목록 확인 중 오류가 발생했으나 계속 진행합니다. ({e})")
    selected_config = None

dataset = None
try:
    # 🚀 시도 1: 스트리밍 모드(streaming=True)로 빠르게 로드 시도 (메모리 효율 최고!)
    print("\n⭐️ 스트리밍 모드 (streaming=True)로 데이터셋을 로드합니다. (가장 빠름!)")
    dataset = load_dataset(DATASET_NAME, split=SPLIT_TO_LOAD, streaming=True)
    print("✅ 스트리밍 로드 성공! 이제 데이터를 메모리 걱정 없이 탐색할 수 있어요.")

except Exception as e:
    # 😭 만약 스트리밍 모드가 어떤 이유로 실패하면? (이럴 때를 대비하는 것이 진짜 프로!)
    print(f"\n⚠️ 경고: 스트리밍 로드 중 오류가 발생했습니다 ({type(e).__name__}).")
    print("➡️ 대안으로, 소량의 데이터를 다운로드하여 진행합니다 (streaming=False).")
    try:
        dataset = load_dataset(DATASET_NAME, split=SPLIT_TO_LOAD, streaming=False)
        print("✅ 일반 다운로드로 데이터셋 로드 성공! 이제 코딩을 시작해봅시다.")
    except Exception as e_fallback:
        print(f"❌ 죄송해요, 데이터 로드에 실패했습니다. 오류: {e_fallback}")
        exit()


# 2. 샘플 데이터 준비
# --------------------------------------------------
print("\n[Step 2] 사용할 샘플 데이터만 추출합니다.")
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)이므로 이 패턴을 사용합니다.
    sample_iterator = dataset.take(SAMPLE_COUNT)
else:
    # 일반 데이터셋 (Dataset)인 경우
    sample_iterator = dataset.take(SAMPLE_COUNT)

# 반복자를 리스트로 변환하여 작업하기 편하게 만듭니다.
sample_data_list = list(sample_iterator)

if not sample_data_list:
    print("⛔️ 로드된 샘플 데이터가 없습니다. 스크립트를 종료합니다.")
    exit()


# 3. 창의적인 AI 실습: 전문 용어 검색 시뮬레이터 구현
# --------------------------------------------------
print("\n=========================================================")
print("✨ [AI 실습 시작] 가상 전문 용어 검색 엔진 테스트!")
print("=========================================================")

def simulate_dictionary_lookup(term_to_check: str, data_samples: list) -> dict:
    """
    주어진 용어(영어)가 데이터셋에 있는지 찾아 가장 적절한 번역을 반환하는 함수입니다.
    (쉽게 말해, 데이터를 검색하는 '뇌' 역할을 해요!)
    """
    print(f"\n🔍 [{term_to_check}] 용어를 검색합니다...")
    
    # 데이터셋을 순회하며 검색하는 척합니다. (반복문 사용법 연습!)
    for i, sample in enumerate(data_samples):
        # 여기서는 'original_term_en' 필드를 기준으로 비교합니다.
        if sample['original_term_en'].strip().lower() == term_to_check.strip().lower():
            return {
                "성공": True,
                "원문 영어": sample['original_term_en'],
                "추정 번역 (중국어):": sample['original_term_zh'],
                "추가 정보 (Source):": sample['source']
            }
        # 만약 데이터가 너무 크다면, 검색 효율을 위해 중간에 멈출 수도 있어요.
        # if i > 5: break # (예시: 5개를 넘으면 멈춤)
            
    return {
        "성공": False,
        "메시지": f"😭 죄송해요! '{term_to_check}'에 대한 번역 기록을 찾지 못했어요. 데이터셋을 더 살펴봐야겠어요."
    }


# ✅ A. 샘플 데이터의 구조 파악 (가장 먼저 해야 할 일!)
print("\n[A] 🔥 데이터 구조 맛보기 (첫 번째 샘플)")
first_sample = sample_data_list[0]
print("-------------------------------------------------------")
print("✨ 데이터 구조:")
print(f"   -> 영어 원문 용어 (original_term_en): {first_sample['original_term_en']}")
print(f"   -> 중국어 번역 (original_term_zh): {first_sample['original_term_zh']}")
print(f"   -> 데이터 출처 (source): {first_sample['source']}")
print("-------------------------------------------------------")


# ✅ B. 용어 검색 시뮬레이션 (가장 재미있는 부분!)
# -------------------------------------------------------
print("\n[B] 🤖 전문 용어 검색 엔진 시뮬레이션 시작!")

# 1. 데이터셋 안에 확실히 있는 용어를 검색해봅시다.
known_term = sample_data_list[0]['original_term_en']
result_known = simulate_dictionary_lookup(known_term, sample_data_list)

print("\n✨ [결과] 검색 성공!")
if result_known['성공']:
    print(f"   ✅ 발견된 용어: {result_known['원문 영어']}")
    print(f"   💡 번역 결과: {result_known['추정 번역 (중국어):']}")
    print(f"   📑 출처 정보: {result_known['추가 정보 (Source)']}")
else:
    print(result_known['메시지'])


# 2. 데이터셋에는 없지만, 우리가 임의로 만든 가짜 용어를 검색해봅시다.
unknown_term = "HyperDimensionalFlux"
result_unknown = simulate_dictionary_lookup(unknown_term, sample_data_list)

print("\n✨ [결과] 검색 실패 (정상 작동!)")
print(f"   ❌ {result_unknown['메시지']}")


# ✅ C. 통계적 관찰 (데이터셋에 어떤 종류의 용어가 많은지 살펴보기)
# --------------------------------------------------
print("\n[C] 📈 데이터셋 통계적 관찰 (사용 가능한 용어 종류 확인)")

# sample_data_list를 사용하여 고유한 (Unique) 영어 용어의 개수를 세어봅시다.
unique_english_terms = set()
for sample in sample_data_list:
    unique_english_terms.add(sample['original_term_en'])

print("-------------------------------------------------------")
print(f"✨ 관찰 결과: 샘플 {SAMPLE_COUNT}개만으로 약 {len(unique_english_terms)}개의 고유한 영어 용어를 확인했습니다.")
print("이것은 실제 데이터셋이 얼마나 다양한 지식을 포함하고 있는지 보여주죠!")
print("=========================================================")

print("\n🎉 축하합니다! 데이터를 성공적으로 로드하고, 분석하고, 창의적으로 활용하는 과정을 모두 마쳤습니다!")
print("데이터셋 분석은 마치 탐험가처럼 끝없는 호기심을 가지고 임하는 것이 중요해요. 오늘도 고생하셨습니다! 👍")